# Figure-Eight Path Demo — Noise Model Experiments

Drives the Nezha firmware through **`SimConnection`** with the full noise model
enabled (encoder noise, motor slip, OTOS noise).  No hardware required.

| # | Experiment | Localisation |
|---|------------|-------------|
| 1 | Dead reckoning | Noisy encoders only — no OTOS correction |
| 2 | OTOS fusion | OTOS corrections reduce drift |
| 3 | Monte Carlo | Noise-sweep → RMS error vs noise level |

**Table of contents**
1. Setup — build `libfirmware_host`, imports, `make_sim()` helper
2. Figure-eight reference path (Catmull-Rom spline)
3. `pure_pursuit_vw` geometry helper
4. Experiment 1 — dead reckoning: encoder drift on the figure-eight

In [1]:
%matplotlib inline
import os
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ.get("PATH", "")
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for notebook execution
import ctypes, math, subprocess, pathlib, sys
import matplotlib.pyplot as plt
import numpy as np

CWD = pathlib.Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "host_tests" else CWD
HOST_DIR = REPO_ROOT / "host"
if str(HOST_DIR) not in sys.path:
    sys.path.insert(0, str(HOST_DIR))

LIB_DIR = REPO_ROOT / "host_tests" / "build"
if not LIB_DIR.exists():
    print("Configuring CMake...")
    subprocess.run(["cmake", "-S", str(REPO_ROOT / "host_tests"), "-B", str(LIB_DIR)],
                   cwd=REPO_ROOT, check=True)

print("Building libfirmware_host...")
sys.stdout.flush()
proc = subprocess.Popen(
    ["cmake", "--build", str(LIB_DIR), "--", "-j4"],
    cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Build failed (exit {proc.returncode})")
print("Build OK")

from robot_radio.io.sim_conn import SimConnection
from robot_radio.robot.protocol import NezhaProtocol
from robot_radio.path.catmull_rom import catmull_rom
from robot_radio.path.path_helper import Path

def make_sim(slip=(0.005, 0.03), enc_noise=0.05, otos_linear=0.01, otos_yaw=0.025):
    """Create a fresh SimConnection + NezhaProtocol with the noise model enabled."""
    conn = SimConnection()
    result = conn.connect()
    if "error" in result:
        raise RuntimeError(result["error"])
    proto = NezhaProtocol(conn)
    proto.set_config(sTimeout=60000)
    conn.set_slip(*slip)
    conn.set_encoder_noise(enc_noise)
    conn.enable_otos_model()
    conn.set_otos_noise(otos_linear, otos_yaw)
    return conn, proto

# Smoke test.
conn_test, proto_test = make_sim()
t_robot, rtt = proto_test.ping()
print(f"PING OK  robot_t={t_robot} ms")
conn_test.disconnect()

Building libfirmware_host...


[100%] Built target firmware_host


Build OK
PING OK  robot_t=0 ms


{'status': 'disconnected', 'ticks': 0}

## 2  Figure-Eight Reference Path

Nine control points trace two lobes of a figure-eight (~600 mm total span).
A centripetal Catmull-Rom spline through these points gives a smooth closed
curve the pure-pursuit tracker can follow.

In [2]:
# Figure-eight control points (mm)
R = 300   # half-span
H = 150   # lobe height

WAYPOINTS_MM = [
    (0, 0),
    (R, H),
    (2*R, 0),
    (R, -H),
    (0, 0),
    (-R, H),
    (-2*R, 0),
    (-R, -H),
    (0, 0),
]

# Generate dense catmull-rom spline
spline_mm = catmull_rom(WAYPOINTS_MM, samples_per_segment=50)
spline_arr = np.array(spline_mm)

# Plot reference path
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(spline_arr[:, 0], spline_arr[:, 1], 'k--', linewidth=1.5, label='Reference path')
ctrl = np.array(WAYPOINTS_MM)
ax.plot(ctrl[:, 0], ctrl[:, 1], 'ko', markersize=5, label='Control points')
ax.set_aspect('equal')
ax.set_xlabel('X (mm)'); ax.set_ylabel('Y (mm)')
ax.set_title('Figure-Eight Reference Path')
ax.legend(); ax.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

print(f"Spline: {len(spline_mm)} points, span x=[{spline_arr[:,0].min():.0f}, {spline_arr[:,0].max():.0f}] mm")

Spline: 401 points, span x=[-600, 600] mm


/var/folders/f_/xrglkf752lxg31tz3b57_fl00000gn/T/ipykernel_21458/473233194.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 3  Pure-Pursuit Helper

`pure_pursuit_vw` implements the standard pure-pursuit geometry:
1. Find the nearest point on the spline to the robot.
2. Walk forward along the spline until cumulative arc length >= `lookahead_mm`.
3. Compute the signed angle `alpha` from robot heading to the goal.
4. `kappa = 2 sin(alpha) / lookahead_mm` → `omega = kappa * v`.

Returns `(v_mms, omega_mrads)` ready for the `VW` firmware command.

In [3]:
def pure_pursuit_vw(spline_pts, pos_mm, yaw_rad, lookahead_mm=150.0, base_speed_mms=200.0):
    """Pure pursuit: returns (v_mms, omega_mrads) for VW command."""
    px, py = pos_mm

    # Find closest point on spline
    pts = np.array(spline_pts)
    dists = np.hypot(pts[:, 0] - px, pts[:, 1] - py)
    nearest_idx = int(np.argmin(dists))

    # Walk forward along spline to find lookahead point
    arc = 0.0
    goal = pts[-1]
    for i in range(nearest_idx + 1, len(pts)):
        seg = np.hypot(pts[i, 0] - pts[i-1, 0], pts[i, 1] - pts[i-1, 1])
        arc += seg
        if arc >= lookahead_mm:
            goal = pts[i]
            break

    # Angle from robot heading to goal in robot frame
    dx, dy = goal[0] - px, goal[1] - py
    angle_to_goal = math.atan2(dy, dx)
    alpha = angle_to_goal - yaw_rad
    # Normalize to [-pi, pi]
    while alpha > math.pi: alpha -= 2 * math.pi
    while alpha < -math.pi: alpha += 2 * math.pi

    # Curvature and omega
    dist_to_goal = math.hypot(dx, dy)
    if dist_to_goal < 1.0:
        return base_speed_mms, 0.0
    kappa = 2.0 * math.sin(alpha) / max(lookahead_mm, 1.0)
    omega_radps = kappa * base_speed_mms  # rad/s (mm/s * 1/mm = rad/s)
    omega_mrads = omega_radps * 1000.0

    # Clamp v and omega
    v_mms = max(50.0, min(base_speed_mms, base_speed_mms))
    omega_mrads = max(-2000.0, min(2000.0, omega_mrads))
    return v_mms, omega_mrads

# Sanity check: robot at origin facing east (+X), goal ahead → omega ≈ 0
v, w = pure_pursuit_vw(spline_mm, (0, 0), 0.0)
print(f"Sanity: v={v:.0f} mm/s, omega={w:.0f} mrad/s (should be near-zero for east-facing robot at origin)")

Sanity: v=200 mm/s, omega=1410 mrad/s (should be near-zero for east-facing robot at origin)


## 4  Experiment 1 — Dead Reckoning

The robot's firmware estimates pose from noisy encoder readings only — no OTOS
correction.  With encoder noise (sigma=0.05 mm/tick) and slip (0.5% straight,
3% extra on turns), dead-reckoning drift accumulates visibly after ~1 loop.

In [4]:
# --- Experiment 1: Dead Reckoning Only ---
conn, proto = make_sim()
conn.clear_state_log()

NUM_STEPS = 500
STEP_MS   = 50
LOOKAHEAD = 150.0
SPEED     = 200.0

truth_log  = []  # exact pose from ExactPoseTracker
est_log    = []  # firmware dead-reckoning from noisy encoders

for step in range(NUM_STEPS):
    state = conn.get_state()
    pos_est = (state["pose_x"], state["pose_y"])
    yaw_est = state["pose_h"]

    v, omega = pure_pursuit_vw(spline_mm, pos_est, yaw_est, LOOKAHEAD, SPEED)
    conn.send(f"VW {int(v)} {int(omega)}")
    conn.send("+")
    conn.tick(STEP_MS)

    truth_log.append(conn.get_exact_pose())
    est_log.append(conn.get_state())

conn.disconnect()

# Plot
truth_x = [p["x"] for p in truth_log]
truth_y = [p["y"] for p in truth_log]
est_x   = [s["pose_x"] for s in est_log]
est_y   = [s["pose_y"] for s in est_log]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(spline_arr[:, 0], spline_arr[:, 1], 'k--', linewidth=1, label='Reference path', alpha=0.6)
ax.plot(truth_x, truth_y, 'g-', linewidth=1.5, label='Ground truth (ExactPoseTracker)')
ax.plot(est_x, est_y, 'r-', linewidth=1.5, label='Dead reckoning (noisy encoders)')
ax.plot(est_x[0], est_y[0], 'go', markersize=10)
ax.set_aspect('equal')
ax.set_xlabel('X (mm)'); ax.set_ylabel('Y (mm)')
ax.set_title('Experiment 1: Dead Reckoning — Encoder Drift')
ax.legend(fontsize=8); ax.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

# Error stats
errors = [math.hypot(truth_log[i]["x"] - est_log[i]["pose_x"],
                     truth_log[i]["y"] - est_log[i]["pose_y"])
          for i in range(len(truth_log))]
print(f"Final position error: {errors[-1]:.1f} mm")
print(f"Mean cross-track error: {sum(errors)/len(errors):.1f} mm")

Final position error: 113.9 mm
Mean cross-track error: 49.7 mm


/var/folders/f_/xrglkf752lxg31tz3b57_fl00000gn/T/ipykernel_21458/564405696.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
